In [ ]:
import os
import glob
import cv2
import matplotlib.pyplot as plt
import tifffile
import pandas as pd
import torch
from torch.utils.data import Dataset
import tifffile
import numpy as np
from torchvision import transforms
import matplotlib.pyplot as plt
import tifffile
import numpy as np
import cv2
import zarr

df = pd.read_csv('../data/processed_data.csv')

BASE_DIR = "../data/gcs/T-CAIREM/"

def get_image_path(nac_id):
    """
    Searches the nested patient folder for their .tif image.
    """
    # This builds a search query: e.g., "gcs/T-CAIREM/NAC-1/*.tif"
    search_pattern = os.path.join(BASE_DIR, str(nac_id), "*.tif")
    
    # glob.glob returns a list of all files that match the pattern
    found_files = glob.glob(search_pattern)
    
    if len(found_files) > 0:
        return found_files[0] # Return the actual file path
    else:
        return None

df['Image_Path'] = df['NAC'].apply(get_image_path)

print(df[['NAC', 'Image_Path']].head())

test_path = df['Image_Path'].dropna().iloc[156]

try:
    with tifffile.TiffFile(test_path) as slide:
        # Get the smallest level (the thumbnail)
        # Often the last page is a 'label' or 'macro' image, 
        # so we try the second to last if the last one looks weird.
        level_to_read = -1 
        
        print(f"Attempting to read Level {len(slide.pages) + level_to_read}...")
        thumbnail = slide.pages[level_to_read].asarray()
        
        print(f"Success! Thumbnail shape: {thumbnail.shape}")
        
        plt.figure(figsize=(8, 8))
        plt.imshow(thumbnail)
        plt.title(f"Patient {df['NAC'].iloc[0]} - WSI Thumbnail")
        plt.axis('off')
        plt.show()

except Exception as e:
    print(f"Error reading slide: {e}")

def extract_tissue_patches(path, patch_size=256, threshold=200, limit=50):
    patches = []
    with tifffile.TiffFile(path) as slide:
        # Access the high-res Level 0
        level_0 = slide.pages[0]
        h, w = level_0.shape[:2]
        
        # Simple grid-based sampling
        for y in range(0, h, h // 20): # Sample 20 points along height
            for x in range(0, w, w // 20): # Sample 20 points along width
                # Read specific tile selection
                patch = level_0.asarray(selection=(y, x, y+patch_size, x+patch_size))
                
                # Simple check: Is the patch mostly white background?
                if np.mean(patch) < threshold: 
                    patches.append(patch)
                
                if len(patches) >= limit: return patches
    return patches


def visualize_patch_extraction_zarr(path, num_patches=4):
    with tifffile.TiffFile(path) as slide:
        # 1. Grab the Thumbnail (Level 5) - Still small and safe
        thumbnail = slide.pages[-1].asarray()
        
        # 2. Open Level 0 as a Zarr array (Memory-mapped)
        # This DOES NOT load the image; it just maps the file
        store = slide.aszarr(series=0, level=0)
        z_array = zarr.open(store, mode='r')
        
        h, w = z_array.shape[:2]
        patches = []
        
        print(f"Slide dimensions: {w}x{h}. Searching for tissue...")
        
        # 3. Search loop
        for _ in range(500):
            sy, sx = np.random.randint(0, h-512), np.random.randint(0, w-512)
            
            # Slicing a Zarr array only reads those specific pixels from disk!
            p = z_array[sy:sy+512, sx:sx+512]
            
            # Simple tissue check (Avoid white background)
            if np.mean(cv2.cvtColor(p, cv2.COLOR_RGB2GRAY)) < 235:
                patches.append(p)
            if len(patches) == num_patches: 
                break

        # 4. Plotting
        fig = plt.figure(figsize=(16, 10))
        grid = plt.GridSpec(2, num_patches, wspace=0.2, hspace=0.3)

        ax_main = fig.add_subplot(grid[0, :])
        ax_main.imshow(thumbnail)
        ax_main.set_title(f"Patient {path.split('/')[-2]} Biopsy Overview", fontsize=14)
        ax_main.axis('off')

        for i, p in enumerate(patches):
            ax = fig.add_subplot(grid[1, i])
            ax.imshow(p)
            ax.set_title(f"High-Res Tissue Patch {i+1}", fontsize=10)
            ax.axis('off')
            
        plt.show()
        return patches


patches = visualize_patch_extraction_zarr(df['Image_Path'].iloc[156])


def visualize_cellular_features(patches_list):
    num_p = len(patches_list)
    fig, axes = plt.subplots(num_p, 3, figsize=(15, 4 * num_p))
    
    for i, patch in enumerate(patches_list):
        # 1. Edge Detection (Cellular Boundaries)
        gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 100, 200) # Highlights membrane/fiber boundaries
        
        # 2. Clump Detection (Nuclear Isolation)
        # We subtract Red from Blue to highlight the purple Hematoxylin stain
        nuclei_data = cv2.subtract(patch[:,:,2], patch[:,:,0])
        # Use a colormap to show density "heat"
        density_map = cv2.applyColorMap(nuclei_data, cv2.COLORMAP_JET)
        density_map = cv2.cvtColor(density_map, cv2.COLOR_BGR2RGB)

        # 3. Plotting
        axes[i, 0].imshow(patch)
        axes[i, 0].set_title(f"Patch {i+1}: Original")
        
        axes[i, 1].imshow(edges, cmap='inferno')
        axes[i, 1].set_title(f"Patch {i+1}: Edges")
        
        axes[i, 2].imshow(density_map)
        axes[i, 2].set_title(f"Patch {i+1}: Nuclear Clumps")
        
        for ax in axes[i]: ax.axis('off')

    plt.tight_layout()
    plt.show()


visualize_cellular_features(patches)

patch_results = []

for i, p in enumerate(patches):
    # 1. Regenerate the features from the raw patch pixels
    gray = cv2.cvtColor(p, cv2.COLOR_RGB2GRAY)
    
    # Isolate nuclei (Purple channel subtraction)
    n_data = cv2.subtract(p[:,:,2], p[:,:,0]) 
    
    # Isolate edges (Canny)
    e_data = cv2.Canny(p, 100, 200)
    
    # 2. Extract metrics
    _, binary_n = cv2.threshold(n_data, 30, 255, cv2.THRESH_BINARY)
    density = np.mean(binary_n > 0)
    complexity = np.mean(e_data > 0)
    
    contours, _ = cv2.findContours(binary_n, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clump_count = len(contours)
    
    # 3. Save with a reference to the patch index
    patch_results.append({
        'Patch_ID': i + 1,
        'Nuclear_Density': density,
        'Edge_Complexity': complexity,
        'Clump_Count': clump_count
    })

# Convert to a DataFrame
image_features_df = pd.DataFrame(patch_results)
print(image_features_df)

patient_metrics = {
    'NAC': 'NAC-50',
    'Avg_Nuclear_Density': image_features_df['Nuclear_Density'].mean(),
    'Max_Clump_Count': image_features_df['Clump_Count'].max(),
    'Avg_Edge_Complexity': image_features_df['Edge_Complexity'].mean()
}

import os
import glob
import tifffile
import zarr
import cv2
import numpy as np
import pandas as pd

# List to store final patient-level metrics
all_patient_features = []

for index, row in df.iterrows():
    nac_id = row['NAC']
    img_path = row['Image_Path'] # We mapped this earlier
    
    if not img_path or not os.path.exists(img_path):
        continue
        
    try:
        with tifffile.TiffFile(img_path) as slide:
            # 1. Open Level 0 via Zarr to stay under RAM limits
            store = slide.aszarr(series=0, level=0)
            z_array = zarr.open(store, mode='r')
            h, w = z_array.shape[:2]
            
            patient_patches = []
            
            # 2. Extract 4 patches containing actual tissue
            attempts = 0
            while len(patient_patches) < 4 and attempts < 100:
                attempts += 1
                sy, sx = np.random.randint(0, h-512), np.random.randint(0, w-512)
                p = z_array[sy:sy+512, sx:sx+512]
                
                # Tissue check: Avoid white space
                if np.mean(cv2.cvtColor(p, cv2.COLOR_RGB2GRAY)) < 235:
                    # 3. Calculate Numerical Info immediately
                    n_data = cv2.subtract(p[:,:,2], p[:,:,0]) # Nuclei isolation
                    e_data = cv2.Canny(p, 100, 200)           # Edge isolation
                    
                    _, binary_n = cv2.threshold(n_data, 30, 255, cv2.THRESH_BINARY)
                    
                    density = np.mean(binary_n > 0)
                    complexity = np.mean(e_data > 0)
                    
                    patient_patches.append({
                        'density': density,
                        'complexity': complexity
                    })

            # 4. Average the patches to get one row per patient
            if patient_patches:
                avg_density = np.mean([x['density'] for x in patient_patches])
                avg_complexity = np.mean([x['complexity'] for x in patient_patches])
                
                all_patient_features.append({
                    'NAC': nac_id,
                    'Img_Nuclear_Density': avg_density,
                    'Img_Edge_Complexity': avg_complexity
                })
                
        print(f"Processed {nac_id}...")
        
    except Exception as e:
        print(f"Error on {nac_id}: {e}")

# Create the new features dataframe
img_features_df = pd.DataFrame(all_patient_features)

img_features_df.head()

final_df = df.merge(img_features_df, on='NAC', how='left')

final_df.head()

print(final_df[['NAC', 'Age', 'ER (%)', 'Img_Nuclear_Density', "NAC Response (0=complete; 1=partial)"]].head())

final_df.to_csv('../data/image_extr_df.csv')

import seaborn as sns
plt.figure(figsize=(14, 6))

# Subplot 1: Nuclear Density vs Response
plt.subplot(1, 2, 1)
sns.boxplot(x="NAC Response (0=complete; 1=partial)", y='Img_Nuclear_Density', data=final_df, palette='Set2')
plt.title('Nuclear Density vs NAC Response\n(0=Complete, 1=Partial)')
plt.grid(axis='y', alpha=0.3)

# Subplot 2: Edge Complexity vs Response
plt.subplot(1, 2, 2)
sns.boxplot(x="NAC Response (0=complete; 1=partial)", y='Img_Edge_Complexity', data=final_df, palette='Set2')
plt.title('Edge Complexity vs NAC Response\n(0=Complete, 1=Partial)')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
# A violin plot shows the distribution of density across different grades
sns.violinplot(x='Grade Nottingham', y='Img_Nuclear_Density', data=final_df, 
               inner='quartile', palette='viridis')
plt.title('Validation: Nuclear Density vs Nottingham Grade')
plt.ylabel('Extracted Nuclear Density')
plt.xlabel('Clinical Grade (Pathologist-assigned)')
plt.show()

import seaborn as sns
import matplotlib.pyplot as plt

# 1. Select the columns you want to compare
# Including your new image metrics and the target 'NAC Response'
cols_to_corr = [
    "NAC Response (0=complete; 1=partial)", 'Img_Nuclear_Density', 'Img_Edge_Complexity', 
    'Age', 'ER (%)', 'PR (%)', 'Grade Nottingham'
]

# 2. Calculate the Spearman correlation matrix
corr_matrix = final_df[cols_to_corr].corr(method='spearman')

# 3. Plotting with specific number formatting
plt.figure(figsize=(12, 10))

sns.heatmap(
    corr_matrix, 
    annot=True,          # This adds the numbers to the squares
    fmt=".2f",           # Limits to 2 decimal places for readability
    cmap='coolwarm',     # Red for positive, Blue for negative correlation
    center=0,            # Ensures 0 is the neutral color
    linewidths=0.5,      # Adds small gaps between the squares
    cbar_kws={"shrink": .8}
)

plt.title('Annotated Correlation Heatmap: Clinical vs. Image Features', fontsize=16)
plt.show()

import seaborn as sns
import matplotlib.pyplot as plt

# Select the most important numerical columns for the matrix
# We include the target 'NAC Response' to color-code the points
cols_for_matrix = [
    'Img_Nuclear_Density', 'Img_Edge_Complexity', 
    'Age', 'ER (%)', "NAC Response (0=complete; 1=partial)"
]

# Create the pairplot
# 'hue' colors the dots by the target: 0 (Complete) vs 1 (Partial)
sns.pairplot(
    final_df[cols_for_matrix], 
    hue="NAC Response (0=complete; 1=partial)", 
    palette='husl',
    diag_kind='kde', # Shows density plots on the diagonal
    plot_kws={'alpha': 0.6, 's': 40} # Makes points slightly transparent
)

plt.suptitle('Scatterplot Matrix: Image Features vs. Clinical Data', y=1.02, fontsize=16)
plt.show()

plt.figure(figsize=(14, 6))

# Subplot 1: Distribution of Nuclear Density
plt.subplot(1, 2, 1)
sns.histplot(data=final_df, x='Img_Nuclear_Density', hue="NAC Response (0=complete; 1=partial)", 
             kde=True, element="step", palette='Set1')
plt.title('Distribution of Nuclear Density by Outcome')
plt.xlabel('Nuclear Density (Clumpiness)')

# Subplot 2: Distribution of Edge Complexity
plt.subplot(1, 2, 2)
sns.histplot(data=final_df, x='Img_Edge_Complexity', hue="NAC Response (0=complete; 1=partial)", 
             kde=True, element="step", palette='Set1')
plt.title('Distribution of Edge Complexity by Outcome')
plt.xlabel('Edge Complexity (Disorganization)')

plt.tight_layout()
plt.show()

# 1. Calculate the Spearman correlation for the entire dataframe

target_cols = ['Img_Nuclear_Density', 'Img_Edge_Complexity', "NAC Response (0=complete; 1=partial)"]
# Spearman is used because 'NAC Response' is a binary category (0 or 1)
correlations = final_df[target_cols].corr(method='spearman')

# 2. Extract just the correlations for your Biomarkers vs. the NAC Response
biomarker_results = correlations.loc[
    ['Img_Nuclear_Density', 'Img_Edge_Complexity'], 
    ["NAC Response (0=complete; 1=partial)"]
]

print(biomarker_results)

# Optional: Check the correlation between the two image features themselves
# This tells you if they are providing unique information
internal_corr = final_df['Img_Nuclear_Density'].corr(final_df['Img_Edge_Complexity'], method='spearman')
print(f"\nInternal Correlation (Density vs. Complexity): {internal_corr:.4f}")

# ── Config ────────────────────────────────────────────────────────────────────
PATCH_SIZE      = 512
STRIDE          = 1024        # was 512
WSI_LEVEL       = 1           # try 2 if still slow; 0 = full res
TISSUE_THRESH   = 235
NUCLEUS_THRESH  = 30
EDGE_LOW, EDGE_HIGH = 100, 200
MIN_CLUMP_AREA  = 50
MAX_PATCHES     = 200         # cap per patient

def build_tissue_mask(z_array, scale=8):
    h, w = z_array.shape[:2]
    step = scale * PATCH_SIZE
    rows = range(0, h - PATCH_SIZE + 1, step)
    cols = range(0, w - PATCH_SIZE + 1, step)
    mask = np.zeros((len(rows), len(cols)), dtype=bool)
    for r, sy in enumerate(rows):
        for c, sx in enumerate(cols):
            tiny = z_array[sy:sy+PATCH_SIZE, sx:sx+PATCH_SIZE]
            if tiny.size == 0: continue
            tiny_small = cv2.resize(tiny, (64, 64))
            if cv2.cvtColor(tiny_small, cv2.COLOR_RGB2GRAY).mean() < TISSUE_THRESH:
                mask[r, c] = True
    return mask

def extract_patch_features(patch):
    n_data = cv2.subtract(patch[:,:,2], patch[:,:,0])
    _, binary_n = cv2.threshold(n_data, NUCLEUS_THRESH, 255, cv2.THRESH_BINARY)
    e_data = cv2.Canny(patch, EDGE_LOW, EDGE_HIGH)
    
    density    = float(np.mean(binary_n > 0))
    complexity = float(np.mean(e_data > 0))
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    closed = cv2.morphologyEx(binary_n, cv2.MORPH_CLOSE, kernel)
    _, _, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    clump_areas = stats[1:, cv2.CC_STAT_AREA]
    clump_areas = clump_areas[clump_areas >= MIN_CLUMP_AREA]
    
    return {
        'density':        density,
        'complexity':     complexity,
        'clump_count':    len(clump_areas),
        'avg_clump_area': float(np.mean(clump_areas)) if len(clump_areas) else 0.0
    }

# ── Main loop ─────────────────────────────────────────────────────────────────
all_patient_features = []

for index, row in df.iterrows():
    nac_id   = row['NAC']
    img_path = row['Image_Path']
    print(f"Starting {nac_id}...", flush=True)
    
    if not img_path or not os.path.exists(img_path):
        print(f"  [{nac_id}] Image not found, skipping.", flush=True)
        continue

    try:
        print(f"  [{nac_id}] Opening slide...", flush=True)
        with tifffile.TiffFile(img_path) as slide:
            n_levels = len(slide.series[0].levels)
            level    = min(WSI_LEVEL, n_levels - 1)  # don't exceed available levels
            store    = slide.aszarr(series=0, level=level)
            z_array  = zarr.open(store, mode='r')
            h, w     = z_array.shape[:2]
            print(f"  [{nac_id}] Slide opened at level {level}: {h}x{w}, building tissue mask...", flush=True)
            
            tissue_mask = build_tissue_mask(z_array)
            thumb_step  = 8 * PATCH_SIZE
            print(f"  [{nac_id}] Tissue mask done ({tissue_mask.sum()}/{tissue_mask.size} cells), scanning grid...", flush=True)
            
            patch_features = []

            for sy in range(0, h - PATCH_SIZE + 1, STRIDE):
                if len(patch_features) >= MAX_PATCHES:
                    break
                for sx in range(0, w - PATCH_SIZE + 1, STRIDE):
                    if len(patch_features) >= MAX_PATCHES:
                        break
                    mr = min(sy // thumb_step, tissue_mask.shape[0] - 1)
                    mc = min(sx // thumb_step, tissue_mask.shape[1] - 1)
                    if not tissue_mask[mr, mc]:
                        continue
                    patch = z_array[sy:sy+PATCH_SIZE, sx:sx+PATCH_SIZE]
                    if patch.shape[:2] != (PATCH_SIZE, PATCH_SIZE):
                        continue
                    if cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY).mean() >= TISSUE_THRESH:
                        continue
                    patch_features.append(extract_patch_features(patch))

            if patch_features:
                all_patient_features.append({
                    'NAC':                       nac_id,
                    'Img_Nuclear_Density':       np.mean([p['density']        for p in patch_features]),
                    'Img_Edge_Complexity':       np.mean([p['complexity']     for p in patch_features]),
                    'Img_Clump_Count_Per_Patch': np.mean([p['clump_count']    for p in patch_features]),
                    'Img_Total_Clump_Count':     sum(   [p['clump_count']    for p in patch_features]),
                    'Img_Avg_Clump_Area':        np.mean([p['avg_clump_area'] for p in patch_features]),
                    'Img_Tissue_Patch_Count':    len(patch_features)
                })
            print(f"  [{nac_id}] Done — {len(patch_features)} patches", flush=True)

    except Exception as e:
        print(f"  [{nac_id}] ERROR: {e}", flush=True)

img_features_df = pd.DataFrame(all_patient_features)
print(f"\nFinished. Features extracted for {len(img_features_df)} patients.")
print(img_features_df.head())

final_df = df.merge(img_features_df, on='NAC', how='left')

final_df.to_csv('../data/full_img_extract.csv')